# Bảng 2 và 3 — GCN của tác giả: train GPT-3.5 Mixed

Notebook này đánh giá các checkpoint GCN của mã nguồn tác giả, đã train trên `gpt3.5_mixed_train_split`.

- **Bảng 2:** test GPT-3.5 Mixed gốc và nhiễu từ đồng nghĩa 5/10/20/30%.
- **Bảng 3:** test HC3 gốc và nhiễu từ đồng nghĩa 5/10/20/30%.
- Nguồn mã được clone từ repository chính thức của tác giả tại commit cố định.
- Chỉ nhận checkpoint đúng định dạng `gpt35_gcn_model_<seed>.pth`.
- Đồ thị được tạo bằng `building_graph.py` của tác giả; RoBERTa chạy trên GPU khi tạo embedding.

Môi trường portable: Python 3.8.20, torch 2.4.1, PyG 2.6.1; không phải môi trường gốc torch 1.11 của tác giả. Không đổi thuật toán GCN hoặc độ chính xác float32.

Có thể chạy lại cell thiết lập sau lỗi mạng: repo và venv hiện có được dùng lại, Git dùng HTTP/1.1 và thử lại tối đa 4 lần. Từng graph được tạo một lần cho tất cả checkpoint, rồi xóa cache để hạn chế dung lượng ổ đĩa.


In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time

WORK = Path('/kaggle/working/author_gcn_gpt35_table2_table3')
AUTHOR = WORK / 'PRDetect'
ENV = WORK / 'python38'
PYTHON = ENV / 'bin' / 'python'
GRAPH = WORK / 'graph_data'
RUN = WORK / 'run'
COMMIT = '83ebc6e2cd543477f83e405d702ac837e664fc9e'
REPO_URL = 'https://github.com/thulx18/PRDetect.git'
CKPT = Path('/kaggle/input/author_gpt35')  # Đổi đường dẫn dataset nếu cần.
if not CKPT.is_dir():
    choices = [d for d in Path('/kaggle/input').iterdir()
               if d.is_dir() and list(d.rglob('gpt35_gcn_model_*.pth'))]
    assert len(choices) == 1, f'Hãy đặt CKPT rõ ràng; tìm thấy {choices}'
    CKPT = choices[0]
CHECKPOINTS = sorted(CKPT.rglob('gpt35_gcn_model_*.pth'))
assert CHECKPOINTS, f'Không có checkpoint GPT-3.5 trong {CKPT}'
seeds = []
for checkpoint in CHECKPOINTS:
    match = re.fullmatch(r'gpt35_gcn_model_(20\d{2})\.pth', checkpoint.name)
    assert match, checkpoint
    seeds.append(int(match.group(1)))
assert len(set(seeds)) == len(seeds), f'Trùng seed: {seeds}'
for directory in (WORK, GRAPH, RUN):
    directory.mkdir(parents=True, exist_ok=True)

def run_retry(command, cwd=None, attempts=4):
    for attempt in range(attempts):
        try:
            return subprocess.run(command, cwd=cwd, check=True)
        except subprocess.CalledProcessError:
            if attempt + 1 == attempts:
                raise
            print(f'Thử lại lần {attempt + 2}/{attempts}: {command[0]}', flush=True)
            time.sleep(3 * (attempt + 1))

# Git HTTP/1.1 tránh lỗi HTTP/2 trên Kaggle. Fetch commit cụ thể thay vì toàn bộ lịch sử.
# Giữ .git khi mạng lỗi để lần chạy sau tiếp tục, không xóa repo hay dữ liệu.
AUTHOR.mkdir(parents=True, exist_ok=True)
if not (AUTHOR / '.git').is_dir():
    assert not any(AUTHOR.iterdir()), f'Thư mục không rỗng và không phải Git repo: {AUTHOR}'
    subprocess.run(['git', 'init', str(AUTHOR)], check=True)
remotes = subprocess.check_output(['git', '-C', str(AUTHOR), 'remote'], text=True).split()
if 'origin' not in remotes:
    subprocess.run(['git', '-C', str(AUTHOR), 'remote', 'add', 'origin', REPO_URL], check=True)
origin = subprocess.check_output(['git', '-C', str(AUTHOR), 'remote', 'get-url', 'origin'], text=True).strip()
assert origin.rstrip('/') == REPO_URL, f'Sai repository: {origin}'
head = subprocess.run(['git', '-C', str(AUTHOR), 'rev-parse', 'HEAD'], capture_output=True, text=True)
if head.returncode != 0 or head.stdout.strip() != COMMIT:
    run_retry(['git', '-c', 'http.version=HTTP/1.1', '-C', str(AUTHOR),
               'fetch', '--depth', '1', 'origin', COMMIT])
    subprocess.run(['git', '-C', str(AUTHOR), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(AUTHOR), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
for source in ('building_graph.py', 'model/GCN2.py'):
    assert (AUTHOR / source).is_file(), source

if shutil.which('uv') is None:
    run_retry([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv')
assert UV, 'Không tìm thấy uv sau khi cài.'
if not PYTHON.is_file():
    run_retry([UV, 'venv', '--python', '3.8.20', str(ENV)])
# Môi trường portable, không phải bản sao chính xác torch 1.11 của tác giả.
run_retry([UV, 'pip', 'install', '--python', str(PYTHON), '--link-mode=copy',
           'numpy==1.24.4', 'scipy==1.10.1', 'torch==2.4.1',
           'torch-geometric==2.6.1', 'transformers==4.45.2', 'spacy==3.7.5',
           'scikit-learn==1.3.2', 'huggingface_hub==0.25.2', 'tqdm==4.66.5'])
probe = subprocess.run([str(PYTHON), '-c', 'import spacy; spacy.load("en_core_web_sm")'],
                       capture_output=True, text=True)
if probe.returncode:
    run_retry([UV, 'pip', 'install', '--python', str(PYTHON), '--link-mode=copy', '--no-deps',
               'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl'])
subprocess.run([str(PYTHON), '-c',
                'import torch,spacy; import torch_geometric; '
                'spacy.load("en_core_web_sm"); '
                'assert torch.cuda.is_available(), "Hãy bật GPU trên Kaggle"; '
                'print("Graph encoder:",torch.cuda.get_device_name(0))'], check=True)
print('Commit:', COMMIT)
print('Checkpoints:', [(seed, p.name) for seed, p in zip(seeds, CHECKPOINTS)])


In [ ]:
%%writefile /kaggle/working/author_gcn_gpt35_table2_table3_run_eval.py
import argparse
import csv
import gc
import hashlib
import importlib.util
import json
import pickle
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import snapshot_download
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torch_geometric.data import Data

parser = argparse.ArgumentParser()
for name in ('author', 'graphs', 'ckpt', 'run'):
    parser.add_argument('--' + name, type=Path, required=True)
a = parser.parse_args()
CASES = [('table2', 'Original', 0, 'gpt3.5_mixed_test_split', 'original_text/gpt3.5_mixed_test_split.json'), ('table2', 'Synonym 5%', 5, 'gpt3.5_mixed_test_adj_5_synonym_replace', 'perturbed_text/gpt3.5_mixed_test_adj_5_synonym_replace.json'), ('table2', 'Synonym 10%', 10, 'gpt3.5_mixed_test_adj_10_synonym_replace', 'perturbed_text/gpt3.5_mixed_test_adj_10_synonym_replace.json'), ('table2', 'Synonym 20%', 20, 'gpt3.5_mixed_test_adj_20_synonym_replace', 'perturbed_text/gpt3.5_mixed_test_adj_20_synonym_replace.json'), ('table2', 'Synonym 30%', 30, 'gpt3.5_mixed_test_adj_30_synonym_replace', 'perturbed_text/gpt3.5_mixed_test_adj_30_synonym_replace.json'), ('table3', 'Original', 0, 'hc3_test', 'original_text/hc3_test.json'), ('table3', 'Synonym 5%', 5, 'hc3_test_adj_5_synonym_replace', 'perturbed_text/hc3_test_adj_5_synonym_replace.json'), ('table3', 'Synonym 10%', 10, 'hc3_test_adj_10_synonym_replace', 'perturbed_text/hc3_test_adj_10_synonym_replace.json'), ('table3', 'Synonym 20%', 20, 'hc3_test_adj_20_synonym_replace', 'perturbed_text/hc3_test_adj_20_synonym_replace.json'), ('table3', 'Synonym 30%', 30, 'hc3_test_adj_30_synonym_replace', 'perturbed_text/hc3_test_adj_30_synonym_replace.json')]
COMMIT = '83ebc6e2cd543477f83e405d702ac837e664fc9e'
assert torch.cuda.is_available(), 'Hãy bật GPU trên Kaggle.'
device = torch.device('cuda')
for directory in (a.graphs, a.run):
    directory.mkdir(parents=True, exist_ok=True)
assert subprocess.check_output(['git', '-C', str(a.author), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT

def sha(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1048576), b''):
            digest.update(block)
    return digest.hexdigest()

# Import đúng file GCN2 của repo vừa tải, không nhầm với module tên model khác.
spec = importlib.util.spec_from_file_location('author_gcn2', a.author / 'model/GCN2.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
checkpoints = []
for path in sorted(a.ckpt.rglob('gpt35_gcn_model_*.pth')):
    match = re.fullmatch(r'gpt35_gcn_model_(20\d{2})\.pth', path.name)
    assert match, path
    # Preflight checkpoint trước khi tốn thời gian tạo graph.
    model = module.GCN2(768, 256, 64)
    model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True), strict=True)
    checkpoints.append((int(match.group(1)), path, sha(path)))
assert checkpoints, 'Không tìm thấy checkpoint GPT-3.5.'
assert len({x[0] for x in checkpoints}) == len(checkpoints), 'Trùng seed.'
del model

for suite, condition, noise, split, relative in CASES:
    source = a.author / relative
    with source.open(encoding='utf8') as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    assert len(records) == 1000, source
    assert all(isinstance(r['text'], str) and r['text'].strip() for r in records), source
    assert {int('human' in r['label']) for r in records} == {0, 1}, source
del records

for attempt in range(4):
    try:
        snapshot_download('FacebookAI/roberta-base',
                          revision='e2da8e2f811d1448a5b465c236feacd80ffbac7b',
                          local_dir=str(a.author / 'roberta-base'),
                          allow_patterns=['config.json', 'pytorch_model.bin', 'vocab.json',
                                          'merges.txt', 'tokenizer_config.json'])
        break
    except Exception:
        if attempt == 3:
            raise
        time.sleep(3 * (attempt + 1))

source = (a.author / 'building_graph.py').read_text(encoding='utf8')
def replace_once(text, old, new):
    assert text.count(old) == 1, f'Mã nguồn tác giả đã thay đổi: {old!r}'
    return text.replace(old, new, 1)

source = replace_once(source, 'import spacy\n', 'import time\nimport spacy\n')
source = replace_once(source, 'vocab_size = len(tokenizer)',
    "vocab_size = len(tokenizer)\ndevice = torch.device('cuda')\n"
    "assert torch.cuda.is_available(), 'Enable GPU'\nmodel = model.to(device)\nmodel.eval()")
source = replace_once(source, 'token_embeddings = last_hidden_states[0]',
                      'token_embeddings = last_hidden_states[0].detach().cpu()')
source = replace_once(source, '/root/autodl-tmp/graph_data', a.graphs.as_posix())
# Tác giả bắt rồi bỏ qua lỗi từng mẫu, có thể làm lệch nhãn; dừng ngay nếu gặp lỗi.
source = replace_once(source, '            print(text)\n            print(e)',
                      '            raise RuntimeError("Graph construction failed") from e')
original_files = 'files = [\n        "hc3_train",\n        "hc3_val",\n        "hc3_test"\n    ]'
assert source.count(original_files) == 1

rows, detail = [], []
metrics = ['accuracy', 'auc', 'f1_human', 'recall_ai', 'recall_human']
out = a.run / 'results'
out.mkdir(exist_ok=True)

def save_results():
    summary = []
    for suite, condition, *_ in CASES:
        group = [r for r in rows if r['suite'] == suite and r['condition'] == condition]
        if not group:
            continue
        row = dict(suite=suite, condition=condition, num_seeds=len(group))
        for metric in metrics:
            values = [r[metric] for r in group]
            row[metric + '_mean'] = float(np.mean(values))
            row[metric + '_std'] = float(np.std(values, ddof=1)) if len(values) > 1 else 0.
        summary.append(row)
    for name, data in [('per_seed_reports.json', detail), ('summary.json', summary)]:
        (out / name).write_text(json.dumps(data, indent=2, allow_nan=False), encoding='utf8')
    fields = ['suite', 'condition', 'noise_percent', 'split', 'seed', 'samples', 'threshold', 'loss'] + metrics
    with (out / 'per_seed_metrics.csv').open('w', newline='', encoding='utf8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows({key: r[key] for key in fields} for r in rows)
    fields = ['suite', 'condition', 'num_seeds'] + [x for m in metrics for x in (m + '_mean', m + '_std')]
    with (out / 'summary.csv').open('w', newline='', encoding='utf8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(summary)

config = dict(author_commit=COMMIT, suites='table2_table3', checkpoint_training_domain='gpt3.5_mixed',
              checkpoint_count=len(checkpoints), threshold=.5, cases=CASES, complete=False,
              environment='Python 3.8.20; torch 2.4.1; PyG 2.6.1 (portable, not original author environment)',
              builder='author building_graph.py; RoBERTa GPU; CPU float32 embeddings; one graph at a time',
              inference='author GCN2/test.py algorithm, batch 1; label 1=human, 0=AI')
(a.run / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf8')

for suite, condition, noise, split, relative in CASES:
    # Không cần copy dữ liệu nhiễu vào checkout; sửa reader để đọc đúng file nguồn.
    patched = replace_once(source, original_files, 'files = [' + repr(split) + ']')
    patched = replace_once(patched, 'f"original_text/{file_name}.json"', repr(str(a.author / relative)))
    builder = a.run / 'author_building_graph_portable.py'
    compile(patched, str(builder), 'exec')
    builder.write_text(patched, encoding='utf8')
    graph = a.graphs / (split + '.pkl')
    # Chỉ ghi/xóa cache có tên xác định trong graph_data của notebook này.
    try:
        subprocess.run([sys.executable, str(builder)], cwd=a.author, check=True)
        with graph.open('rb') as stream:
            g = pickle.load(stream)
        y = g['y'].cpu().numpy().astype(int)
        assert len(y) == len(g['all_token_embeddings']) == len(g['all_edge_index']) == 1000
        assert set(y) == {0, 1}
        for seed, path, checkpoint_sha in checkpoints:
            model = module.GCN2(768, 256, 64).to(device)
            model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True), strict=True)
            model.eval()
            probabilities = []
            with torch.inference_mode():
                for x, edge, label in zip(g['all_token_embeddings'], g['all_edge_index'], g['y']):
                    assert x.ndim == 2 and x.shape[1] == 768 and x.shape[0] > 0
                    assert edge.ndim == 2 and edge.shape[0] == 2
                    assert edge.numel() == 0 or (int(edge.min()) >= 0 and int(edge.max()) < len(x))
                    datum = Data(x=x, edge_index=edge, y=label).to(device)
                    probabilities.append(model(datum).item())
            q = np.asarray(probabilities, dtype=np.float32)
            assert np.isfinite(q).all() and ((q >= 0) & (q <= 1)).all()
            pred = (q >= .5).astype(int)
            cm = [[int(((y == i) & (pred == j)).sum()) for j in (0, 1)] for i in (0, 1)]
            row = dict(suite=suite, condition=condition, noise_percent=noise, split=split, seed=seed,
                       checkpoint=path.name, checkpoint_sha256=checkpoint_sha, samples=len(y), threshold=.5,
                       loss=float(torch.nn.functional.binary_cross_entropy(torch.tensor(q), torch.tensor(y, dtype=torch.float32))),
                       accuracy=float(accuracy_score(y, pred)), auc=float(roc_auc_score(y, q)),
                       f1_human=float(f1_score(y, pred, zero_division=0)), recall_ai=cm[0][0]/sum(cm[0]),
                       recall_human=cm[1][1]/sum(cm[1]), confusion_matrix_rows_true_columns_pred=cm)
            rows.append(row)
            detail.append(dict(row, y_true=y.tolist(), p_human=q.tolist()))
            save_results()  # Giữ kết quả từng seed nếu kết nối hoặc lượt sau bị lỗi.
            print(suite, condition, 'seed', seed, 'accuracy', row['accuracy'], 'auc', row['auc'], flush=True)
            del model, datum
            torch.cuda.empty_cache()
        del g, x, edge, label
        gc.collect()
    finally:
        if graph.is_file():
            graph.unlink()  # Cache tái tạo được; không xóa checkpoint hoặc kết quả.

assert len(rows) == len(checkpoints) * len(CASES)
config['complete'] = True
(a.run / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf8')
print('finished', len(checkpoints), 'checkpoints x', len(CASES), 'conditions', flush=True)


In [ ]:
# Chạy bằng subprocess để nhận chính xác mã lỗi và không phụ thuộc _exit_code của IPython.
subprocess.run([str(PYTHON), '/kaggle/working/author_gcn_gpt35_table2_table3_run_eval.py',
                '--author', str(AUTHOR), '--graphs', str(GRAPH),
                '--ckpt', str(CKPT), '--run', str(RUN)], cwd=AUTHOR, check=True)
print((RUN / 'results' / 'summary.csv').read_text(encoding='utf8'))


In [ ]:
import json
import shutil
from IPython.display import FileLink, display

config = json.loads((RUN / 'run_config.json').read_text(encoding='utf8'))
assert config['complete'], 'Đánh giá chưa hoàn tất; xem kết quả tạm trong run/results.'
archive = Path('/kaggle/working/author_gcn_gpt35_table2_table3_results')
shutil.make_archive(str(archive), 'zip', RUN)
print(f'Đã tạo: {archive}.zip')
display(FileLink(str(archive) + '.zip'))
